In [ ]:
import torch 
import torch.nn as nn
import torch.utils.data as data
import torch.optim as optim
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import pandas as pd
import numpy as np

In [ ]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
num_epochs = 20
batch_size = 32
learning_rate = 0.001

In [ ]:
class MNISTDataset(data.Dataset):
    def __init__(self, train=True, transform=transforms.Compose([transforms.ToTensor()])):
        super().__init__()
        self.transform = transform
        self.train = train
        if self.train:
            data = pd.read_csv('../input/digit-recognizer/train.csv').to_numpy()
            self.images = data[:, 1:].reshape((-1, 28, 28, 1)) / 255.0
            self.labels = data[:, 0]
        else:
            data = pd.read_csv('../input/digit-recognizer/test.csv').to_numpy()
            self.images = data.reshape((-1, 28, 28, 1)) / 255.0

    def __getitem__(self, index):
        img = self.images[index]
        img = self.transform(img)
        if self.train:
            label = self.labels[index]
            return img, label
        else:
            return img   

    def __len__(self):
        return self.images.shape[0]
    

train_dataset = MNISTDataset()
test_dataset = MNISTDataset(train=False)
train_loader = data.DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
test_loader = data.DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super(ConvBlock, self).__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding='same'),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(),
            nn.Conv2d(out_ch, out_ch, 3, padding='same'),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(),
            nn.MaxPool2d(2))

    def forward(self, x):
        return self.block(x)

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        self.conv1 = ConvBlock(1, 32)
        self.conv2 = ConvBlock(32, 64)
        self.fc = nn.Linear(3136, 10)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = x.reshape(x.size(0), -1)
        x = self.fc(x)
        return x

model = CNN().to(device)

In [ ]:
model.train()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)
for epoch in range(num_epochs):
    epoch_loss = 0
    for i, (images, labels) in enumerate(train_loader):
        images = images.to(device).float()
        labels = labels.to(device)
        outputs = model(images)
        
        loss = criterion(outputs, labels)
        epoch_loss += loss.item()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
    print('epoch: ', epoch, 'loss: ', epoch_loss)

In [ ]:
model.eval()  
with torch.no_grad():
    correct = 0
    total = 0
    for images, labels in train_loader:
        images = images.to(device).float()
        labels = labels.to(device)
        outputs = model(images)
        predicted = torch.argmax(outputs, axis=1)
        total += labels.shape[0]
        correct += (predicted == labels).sum().item()

    print(correct / total)

In [ ]:
sample_sub = pd.read_csv('../input/digit-recognizer/sample_submission.csv')
model.eval()  
with torch.no_grad():
    total = 0
    for images in test_loader:
        images = images.to(device).float()
        outputs = model(images)
        predicted = torch.argmax(outputs, axis=1)
        samples = predicted.shape[0]
        sample_sub.loc[total: total + samples - 1, 'Label'] = predicted.cpu().numpy()
        total += samples

In [ ]:
sample_sub.to_csv('submission.csv',index=False)